# Build the first modeling table (auto-picks best-covered drug)

In [ ]:
from pathlib import Path
import os

# set project root = parent of /notebooks
PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)

print("CWD set to:", Path.cwd())

load processed files

In [ ]:
from pathlib import Path
import pandas as pd

OUT = Path("data/processed")

expr = pd.read_parquet(OUT / "expression_tpm_logp1.parquet")
pr = pd.read_parquet(OUT / "prism_secondary_collapsed.parquet")

expr.shape, pr.shape

find the best-covered compound

In [ ]:
coverage = (
    pr.groupby(["broad_id", "compound_name"])["depmap_id"]
      .nunique()
      .reset_index(name="n_cell_lines")
      .sort_values("n_cell_lines", ascending=False)
)

coverage.head(20)

merge to make modeling table (AUC)

In [ ]:
top = coverage.iloc[0]
broad_id = top["broad_id"]
compound = top["compound_name"]
target_col = "auc"

print("Selected:", compound, "|", broad_id, "| n_cell_lines:", int(top["n_cell_lines"]))

pr_drug = pr.loc[pr["broad_id"] == broad_id, ["depmap_id", target_col]].dropna()
df_model = pr_drug.merge(expr, on="depmap_id", how="inner")

print("Modeling table shape:", df_model.shape)
df_model.head()

save modeling table

In [ ]:
safe_name = "".join(ch if ch.isalnum() else "_" for ch in compound.lower()).strip("_")
out_model = OUT / f"modeling_{safe_name}_{target_col}.parquet"
df_model.to_parquet(out_model, index=False)

print("Wrote:", out_model)